# Aria Integration & Extensibility Guide

**Complete guide for extending Aria with new features, integrating third-party services, and building plugins.**

Learn how to add new capabilities to Aria.


## Extension Architecture

### Plugin System

```
Aria Core
├─ API Layer
├─ AI Chat Providers
├─ Database Layer
└─ Message Routing

Plugins
├─ Chat Providers (LMStudio, Ollama, OpenAI)
├─ Speech Providers (Azure TTS, Google TTS)
├─ Embedding Providers (OpenAI, Hugging Face)
├─ Database Backends (PostgreSQL, Cosmos)
├─ Payment Providers (Stripe, PayPal)
└─ Monitoring Providers (Datadog, New Relic)
```

### Extension Points

```python
# 1. Chat Provider Interface
class ChatProvider(ABC):
    @abstractmethod
    async def complete(self, prompt: str) -> str:
        """Generate text completion."""
        pass

    @abstractmethod
    async def stream(self, prompt: str) -> AsyncGenerator[str, None]:
        """Stream text generation."""
        pass

# 2. Embedding Provider Interface
class EmbeddingProvider(ABC):
    @abstractmethod
    async def embed(self, text: str) -> list[float]:
        """Generate embeddings."""
        pass

# 3. Database Backend Interface
class DatabaseBackend(ABC):
    @abstractmethod
    async def query(self, sql: str) -> list[dict]:
        """Execute query."""
        pass

    @abstractmethod
    async def execute(self, sql: str, params: dict) -> int:
        """Execute statement."""
        pass

# 4. TTS Provider Interface
class TTSProvider(ABC):
    @abstractmethod
    async def synthesize(self, text: str, voice: str) -> bytes:
        """Synthesize speech."""
        pass
```

---

## Creating a Custom Chat Provider

### Step 1: Implement Provider Interface

```python
# ai_projects/chat-cli/src/custom_provider.py
import asyncio
from typing import AsyncGenerator
from shared.chat_providers import ChatProvider

class CustomChatProvider(ChatProvider):
    """Custom chat provider integration."""

    def __init__(self, api_key: str, model: str = "custom-model"):
        self.api_key = api_key
        self.model = model
        self.base_url = "https://api.custom-service.com/v1"

    async def complete(self, prompt: str) -> str:
        """Generate text completion."""
        import aiohttp

        async with aiohttp.ClientSession() as session:
            payload = {
                "model": self.model,
                "prompt": prompt,
                "max_tokens": 100,
                "temperature": 0.7
            }

            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            }

            async with session.post(
                f"{self.base_url}/completions",
                json=payload,
                headers=headers
            ) as resp:
                result = await resp.json()
                return result["choices"][0]["text"]

    async def stream(self, prompt: str) -> AsyncGenerator[str, None]:
        """Stream text generation."""
        import aiohttp

        async with aiohttp.ClientSession() as session:
            payload = {
                "model": self.model,
                "prompt": prompt,
                "stream": True,
                "max_tokens": 100
            }

            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            }

            async with session.post(
                f"{self.base_url}/completions",
                json=payload,
                headers=headers
            ) as resp:
                async for line in resp.content:
                    if line:
                        data = line.decode().strip()
                        if data.startswith("data: "):
                            text = data[6:]
                            yield text

    @property
    def name(self) -> str:
        return "custom-provider"

    async def validate_connection(self) -> bool:
        """Check if provider is available."""
        try:
            response = await self.complete("test")
            return bool(response)
        except Exception as e:
            print(f"Provider validation failed: {e}")
            return False
```

### Step 2: Register Provider

```python
# shared/chat_providers.py (modified to add custom provider)
from custom_provider import CustomChatProvider

REGISTERED_PROVIDERS = {
    "openai": OpenAIProvider,
    "azure": AzureOpenAIProvider,
    "lmstudio": LMStudioProvider,
    "ollama": OllamaProvider,
    "custom": CustomChatProvider,  # New provider
    "local": LocalProvider
}

def detect_provider(provider_name: str = "auto") -> ChatProvider:
    """Detect and instantiate provider."""
    if provider_name == "custom":
        api_key = os.getenv("CUSTOM_PROVIDER_API_KEY")
        if not api_key:
            raise ValueError("CUSTOM_PROVIDER_API_KEY not set")
        return CustomChatProvider(api_key)

    # ... existing detection logic
```

### Step 3: Configure Environment

```bash
# .env or local.settings.json
CUSTOM_PROVIDER_API_KEY=sk-...
DEFAULT_AI_PROVIDER=custom
```

### Step 4: Test Provider

```bash
# Test from command line
python -m ai-projects/chat-cli/src/chat_cli.py \
  --provider custom \
  --once "Hello, what is your name?"

# Expected output:
# I'm a custom provider instance. How can I help?
```

---

## Integrating External Services

### Stripe Payment Integration

```python
# shared/payment_providers/stripe_provider.py
import stripe
from typing import Optional

class StripePaymentProvider:
    def __init__(self, api_key: str):
        stripe.api_key = api_key

    def create_payment_intent(
        self,
        amount: int,  # cents
        currency: str = "usd",
        customer_id: Optional[str] = None
    ) -> str:
        """Create Stripe payment intent."""
        intent = stripe.PaymentIntent.create(
            amount=amount,
            currency=currency,
            customer=customer_id
        )
        return intent.client_secret

    def create_subscription(
        self,
        customer_id: str,
        price_id: str
    ) -> dict:
        """Create recurring subscription."""
        subscription = stripe.Subscription.create(
            customer=customer_id,
            items=[{"price": price_id}]
        )
        return {
            "id": subscription.id,
            "status": subscription.status,
            "current_period_end": subscription.current_period_end
        }

    def handle_webhook(self, event: dict) -> None:
        """Handle Stripe webhook events."""
        event_type = event["type"]

        if event_type == "payment_intent.succeeded":
            payment_intent = event["data"]["object"]
            # Handle successful payment
            print(f"✓ Payment succeeded: {payment_intent['id']}")

        elif event_type == "customer.subscription.updated":
            subscription = event["data"]["object"]
            # Handle subscription update
            print(f"✓ Subscription updated: {subscription['id']}")

# Integration in function_app.py
@app.route("/api/payments/create-intent", methods=["POST"])
async def create_payment_intent(req: func.HttpRequest):
    """Create payment intent."""
    amount = req.get_json().get("amount")

    stripe_provider = StripePaymentProvider(
        api_key=os.getenv("STRIPE_API_KEY")
    )

    client_secret = stripe_provider.create_payment_intent(amount)
    return func.HttpResponse(
        json.dumps({"clientSecret": client_secret}),
        status_code=200
    )
```

### Slack Integration

```python
# shared/integrations/slack_notifier.py
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError

class SlackNotifier:
    def __init__(self, bot_token: str):
        self.client = WebClient(token=bot_token)

    def send_message(
        self,
        channel: str,
        text: str,
        blocks: Optional[list] = None
    ) -> bool:
        """Send Slack message."""
        try:
            self.client.chat_postMessage(
                channel=channel,
                text=text,
                blocks=blocks
            )
            return True
        except SlackApiError as e:
            print(f"Slack error: {e}")
            return False

    def send_alert(self, severity: str, message: str) -> bool:
        """Send alert to #alerts channel."""
        color = {
            "critical": "#FF0000",
            "warning": "#FFA500",
            "info": "#0000FF"
        }.get(severity, "#000000")

        blocks = [
            {
                "type": "section",
                "text": {
                    "type": "mrkdwn",
                    "text": f"*{severity.upper()}*\n{message}"
                }
            }
        ]

        return self.send_message("#alerts", message, blocks)

# Usage
slack = SlackNotifier(os.getenv("SLACK_BOT_TOKEN"))
slack.send_alert("warning", "Model accuracy dropped below threshold")
```

### GitHub Integration

```python
# shared/integrations/github_notifier.py
from github import Github, GithubException

class GitHubIntegration:
    def __init__(self, token: str, repo: str):
        self.github = Github(token)
        self.repo = self.github.get_repo(repo)

    def create_issue(self, title: str, body: str, labels: list[str] = None) -> str:
        """Create GitHub issue."""
        issue = self.repo.create_issue(
            title=title,
            body=body,
            labels=labels or []
        )
        return issue.html_url

    def create_pull_request(
        self,
        title: str,
        body: str,
        head: str,
        base: str = "main"
    ) -> str:
        """Create pull request."""
        pr = self.repo.create_pull(
            title=title,
            body=body,
            head=head,
            base=base
        )
        return pr.html_url

    def add_label_to_issue(self, issue_number: int, label: str) -> None:
        """Add label to issue."""
        issue = self.repo.get_issue(issue_number)
        issue.add_to_labels(label)

# Usage
github = GitHubIntegration(
    token=os.getenv("GITHUB_TOKEN"),
    repo="Bryan-Roe/Aria"
)

# Report error
github.create_issue(
    title="Model accuracy degradation",
    body="Model accuracy dropped 5% in last evaluation",
    labels=["bug", "ml"]
)
```

---

## Building Hooks & Middleware

### Middleware Pattern

```python
# shared/middleware.py
from typing import Callable, Awaitable
from functools import wraps

class MiddlewareChain:
    """Chain middleware handlers."""

    def __init__(self):
        self.middlewares = []

    def add(self, middleware: Callable):
        """Add middleware to chain."""
        self.middlewares.append(middleware)
        return self

    async def execute(self, handler: Callable, request: dict):
        """Execute middleware chain."""
        async def next_middleware(index: int):
            if index >= len(self.middlewares):
                return await handler(request)

            middleware = self.middlewares[index]
            return await middleware(request, lambda: next_middleware(index + 1))

        return await next_middleware(0)

# Define custom middleware
async def auth_middleware(request: dict, next_handler: Callable):
    """Check authentication."""
    token = request.headers.get("Authorization")
    if not token:
        raise Exception("Unauthorized")

    request["user"] = verify_token(token)
    return await next_handler()

async def rate_limit_middleware(request: dict, next_handler: Callable):
    """Rate limiting."""
    user_id = request.get("user_id")
    if is_rate_limited(user_id):
        raise Exception("Rate limit exceeded")

    return await next_handler()

async def logging_middleware(request: dict, next_handler: Callable):
    """Request logging."""
    print(f"→ {request.get('method')} {request.get('path')}")
    result = await next_handler()
    print(f"← {request.get('path')} OK")
    return result

# Build middleware chain
chain = MiddlewareChain()
chain.add(logging_middleware)
chain.add(auth_middleware)
chain.add(rate_limit_middleware)

# Use in handler
async def chat_handler(request: dict):
    return await chain.execute(handle_chat_request, request)
```

### Request Hooks

```python
# shared/hooks.py
class RequestHooks:
    """Lifecycle hooks for requests."""

    def __init__(self):
        self.before_hooks = []
        self.after_hooks = []
        self.error_hooks = []

    def before(self, fn: Callable) -> None:
        """Hook before request processing."""
        self.before_hooks.append(fn)

    def after(self, fn: Callable) -> None:
        """Hook after request processing."""
        self.after_hooks.append(fn)

    def error(self, fn: Callable) -> None:
        """Hook on error."""
        self.error_hooks.append(fn)

    async def execute(self, handler: Callable, request: dict):
        """Execute with hooks."""
        try:
            # Before hooks
            for hook in self.before_hooks:
                await hook(request)

            # Main handler
            result = await handler(request)

            # After hooks
            for hook in self.after_hooks:
                await hook(request, result)

            return result

        except Exception as e:
            # Error hooks
            for hook in self.error_hooks:
                await hook(request, e)
            raise

# Usage
hooks = RequestHooks()

@hooks.before
async def validate_input(request: dict):
    """Validate input before processing."""
    if "message" not in request:
        raise ValueError("Missing message field")

@hooks.after
async def log_metrics(request: dict, result):
    """Log metrics after processing."""
    print(f"Response size: {len(str(result))}")

@hooks.error
async def on_error(request: dict, error):
    """Handle errors."""
    print(f"Error: {error}")
```


## Extension Packaging

### Creating a Plugin Package

```python
# my_extension/setup.py
from setuptools import setup

setup(
    name="aria-custom-provider",
    version="1.0.0",
    description="Custom chat provider for Aria",
    packages=["aria_custom_provider"],
    install_requires=[
        "aiohttp>=3.8.0",
        "pydantic>=2.0.0"
    ],
    entry_points={
        "aria.chat_providers": [
            "custom = aria_custom_provider:CustomChatProvider"
        ]
    }
)

# Install as editable
pip install -e ./my_extension

# Or from PyPI
pip install aria-custom-provider
```

### Plugin Discovery

```python
# shared/plugin_loader.py
import importlib.metadata
from typing import Dict, Type

def discover_plugins(group: str) -> Dict[str, Type]:
    """Discover plugins by entry point."""
    plugins = {}

    for entry_point in importlib.metadata.entry_points().get(f"aria.{group}", []):
        try:
            plugin_class = entry_point.load()
            plugins[entry_point.name] = plugin_class
            print(f"✓ Loaded plugin: {entry_point.name}")
        except Exception as e:
            print(f"❌ Failed to load plugin {entry_point.name}: {e}")

    return plugins

# Usage
chat_providers = discover_plugins("chat_providers")
for name, provider_class in chat_providers.items():
    print(f"Available: {name}")
```

---

## Extension Best Practices

### Testing Extensions

```python
# tests/test_custom_provider.py
import pytest
from custom_provider import CustomChatProvider

@pytest.fixture
def provider():
    return CustomChatProvider(api_key="test-key")

@pytest.mark.asyncio
async def test_complete(provider):
    """Test completion."""
    response = await provider.complete("Hello")
    assert isinstance(response, str)
    assert len(response) > 0

@pytest.mark.asyncio
async def test_stream(provider):
    """Test streaming."""
    chunks = []
    async for chunk in provider.stream("Hello"):
        chunks.append(chunk)

    assert len(chunks) > 0
    full_response = "".join(chunks)
    assert len(full_response) > 0

@pytest.mark.asyncio
async def test_validation(provider):
    """Test provider validation."""
    is_valid = await provider.validate_connection()
    assert isinstance(is_valid, bool)
```

### Documentation

````markdown
# Custom Provider for Aria

## Installation

```bash
pip install aria-custom-provider
```
````

## Configuration

Set environment variables:

```
CUSTOM_PROVIDER_API_KEY=sk-...
DEFAULT_AI_PROVIDER=custom
```

## Usage

```python
from custom_provider import CustomChatProvider

provider = CustomChatProvider(api_key="sk-...")
response = await provider.complete("Hello")
```

## API Reference

### `complete(prompt: str) -> str`

Generate text completion for the given prompt.

### `stream(prompt: str) -> AsyncGenerator[str, None]`

Stream text generation.

```

---

## Extension Versioning

### Semantic Versioning

```

MAJOR.MINOR.PATCH
↑ ↑ ↑
│ │ └─ Bug fixes (1.0.1)
│ └────────── Features, backward compatible (1.1.0)
└───────────────── Breaking changes (2.0.0)

Examples:
1.0.0 → 1.0.1 (patch: small fixes)
1.0.0 → 1.1.0 (minor: new features)
1.0.0 → 2.0.0 (major: breaking changes)

````

### Compatibility Matrix

```yaml
# pyproject.toml
aria-custom-provider:
  "1.0.0":
    compatible_with: ["Aria 1.0+"]
    requires: ["aiohttp>=3.8.0"]

  "2.0.0":
    compatible_with: ["Aria 2.0+"]
    requires: ["aiohttp>=3.9.0"]
    breaking_changes:
      - "ChatProvider interface changed"
      - "Removed sync methods"
````
